In [2]:
import random
import math

from functools import reduce
from itertools import combinations

Consts and Setup

In [3]:
eps = 1.0e-14
N = 16
seed = 42

In [4]:
random.seed(seed)

The Heart of Darkness

In [ ]:
class DifferentialEvolution:
    def __init__(self,
                 objective_function,
                 constraint_functions,
                 upper_bounds,
                 lower_bounds,
                 population_size):
        self.objective_function = objective_function
        self.constraint_functions = constraint_functions
        self.upper_bounds = upper_bounds
        self.lower_bounds = lower_bounds
        self.population_size = population_size

    def run(self,
            stopping_condition,
            crossover,
            selection_mutation):
        self.initialize_stats()

        population = [[self.new_chromosome(upper, lower) for upper, lower in zip(self.upper_bounds, self.lower_bounds)] for _ in range(self.population_size)]
        scores = [self.evaluate(x) for x in population]
        self.update_stats(population, scores)

        while stopping_condition.check(self):
            new_generation = []
            new_scores = []

            for x, s in zip(population, scores):
                u = crossover.do(x)
                v = selection_mutation.do(x,
                                          population,
                                          self.best_solution_hist[-1],
                                          u)
                v = self.enforce_bounds(v)
                f = self.evaluate(v)

                if f < s:
                    new_generation.append(v)
                    new_scores.append(s)
                else:
                    new_generation.append(x)
                    new_scores.append(s)

            population = new_generation
            scores = new_scores
            self.iter_count += 1
            self.update_stats(population, scores)
        
        return self

    def initialize_stats(self):
        self.iter_count = 0
        self.eval_count = 0
        self.best_solution_hist = []
        self.best_score_hist = []
        self.population_diversity_hist = []

    def update_stats(self, population, scores):
        best_idx = scores.index(min(scores))

        self.best_solution_hist.append(population[best_idx])
        self.best_score_hist.append(scores[best_idx])
        self.population_diversity_hist.append(self.diversity(population))

    def new_chromosome(self, upper, lower):
        return lower + random.random()*(upper - lower)
    
    def enforce_bounds(self, x):
        return [a if lower<=a and a<=upper else self.new_chromosome(upper, lower) for a, upper, lower in zip(x, self.upper_bounds, self.lower_bounds)]
    
    def evaluate(self, x):
        self.eval_count += 1

        cs = [cf(x) for cf in self.constraint_functions]
        cs = [c if eps < c else 1 for c in cs]

        return self.objective_function(x) * reduce(lambda x, y: x * y, cs)

    def diversity(self, population):
        n = self.population_size
        total_dist = 0
        
        for a, b in combinations(population, r=2):
            total_dist += math.sqrt(sum([(x - y)**2 for x, y in zip(a, b)]))

        return total_dist/(n*(n-1)/2)

Stopping Conditions

In [ ]:
class MaxIterationsStop:
    def __init__(self,
                 max_iterations):
        self.max_iterations = max_iterations
    
    def check(self, record):
        return record.iter_count < self.max_iterations

In [ ]:
class FitnessThresholdStop:
    def __init__(self,
                 fitness_threshold):
        self.fitness_threshold = fitness_threshold
    
    def check(self, record):
        return self.fitness_threshold < record.best_score_hist[-1]

In [ ]:
class NoImprovementStop:
    def __init__(self,
                 max_no_improve_iters,
                 no_improve_threshold):
        self.max_no_improve_iters = max_no_improve_iters
        self.no_improve_threshold = no_improve_threshold
    
    def check(self, record):
        if self.max_no_improve_iters < record.iter_count:
            window = record.best_score_hist[-self.max_no_improve_iters:]
            improves = [abs(b - a) for a, b in zip(window[:-1], window[1:])]
            return not all([imp < self.no_improve_threshold for imp in improves])
        return True

Crossovers

In [ ]:
class Crossover:
    def __init__(self, p):
        self.p = p
    
    def do(self, x):
        d = random.randrange(len(x))
        u = [self.p < random.random() for _ in x]
        u[d] = True

        return u

In [ ]:
class CrossoverSA:
    def do(self, x):
        d = random.randrange(len(x))
        u = [random.random() < random.gauss(0.5, 0.15) for _ in x]
        u[d] = True

        return u

Selections and Mutations

In [ ]:
class SelectMutateRand1:
    def __init__(self,
                 omega):
        self.omega = omega

    def do(self,
           xx,
           population,
           _,
           uu):
        aa, bb, cc = random.sample(population, 3)

        return [a + self.omega*(b - c) if u else x for a, b, c, u, x in zip(aa, bb, cc, uu, xx)]

In [18]:
class SelectMutateBest2:
    def __init__(self,
                 nu,
                 omega):
        self.nu = nu
        self.omega = omega

    def do(self,
           xx,
           population,
           best,
           uu):
        aa, bb, cc, dd = random.sample(population, 4)

        return [bst + self.nu*(a - b) + self.omega*(c - d) if u else x for a, b, c, d, u, x, bst in zip(aa, bb, cc, dd, uu, xx, best)]
        

In [ ]:
class SelectMutateSDE:
    def __init__(self,
                 dimension):
        self.omegas = [random.gauss(0.5, 0.15) for _ in range(dimension)]
    
    def do(self,
           xx,
           population,
           _,
           uu):
        self.omegas = [self.__evolve_omega() for _ in self.omegas]
        aa, bb, cc = random.sample(population, 3)

        return [a + o*(b - c) if u else x for a, b, c, o, u, x in zip(aa, bb, cc, self.omegas, uu, xx)]

    def __evolve_omega(self):
        o1, o2, o3 = random.sample(self.omegas, 3)

        return o1 + random.gauss(0, 0.5)*(o2 - o3)

Benchmark Testbed